<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 6: Siniflandirma Temel

**MAKİNE ÖĞRENMESİ UZMANLIĞI** · Modül 6 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta06/hafta06_siniflandirma_temel.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta06/hafta06_siniflandirma_temel.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>

</div>

# Hafta 6 — Sınıflandırma Temelleri

Bu defterde ikili (binary) sınıflandırma kavramını öğrenecek, **Lojistik Regresyon**, **K-En Yakın Komşu (KNN)** ve **Karar Ağacı** algoritmalarını scikit-learn ile uygulayacağız.

## İçindekiler
1. İkili Sınıflandırma Kavramı
2. Kütüphaneler ve Veri Hazırlığı (Iris Veri Seti)
3. Model Eğitimi: Lojistik Regresyon, KNN, Karar Ağacı
4. Karışıklık Matrisi ve Sınıflandırma Raporu
5. ROC Eğrisi ve AUC
6. Çapraz Doğrulama ile Model Karşılaştırması
7. Karar Sınırları Görselleştirmesi (2D)

---

## 1. İkili Sınıflandırma Kavramı

**Sınıflandırma**, gözetimli öğrenmenin temel görevlerinden biridir. Amaç, giriş verilerine bakarak bir **sınıf etiketi** tahmin etmektir.

- **İkili (Binary) Sınıflandırma:** Sadece iki sınıf vardır (Evet/Hayır, Hasta/Sağlıklı, Spam/Spam değil)
- **Çoklu Sınıflandırma:** İkiden fazla sınıf bulunur (Iris türleri: setosa, versicolor, virginica)

### Temel Metrikler
| Metrik | Açıklama |
|--------|----------|
| **Doğruluk (Accuracy)** | Doğru tahminlerin toplam tahminlere oranı |
| **Kesinlik (Precision)** | Pozitif tahminlerin ne kadarı gerçekten pozitif |
| **Duyarlılık (Recall)** | Gerçek pozitiflerin ne kadarı doğru tahmin edildi |
| **F1 Skoru** | Kesinlik ve duyarlılığın harmonik ortalaması |
| **AUC** | ROC eğrisi altındaki alan |

## 2. Kütüphaneler ve Veri Hazırlığı

### 2.1 Kütüphanelerin Yüklenmesi

Bu projede kullanacağımız temel kütüphaneler:

| Kütüphane | Amacı |
|-----------|-------|
| `sklearn.datasets` | Hazır veri setleri (Iris) |
| `sklearn.model_selection` | Eğitim/test ayırma ve çapraz doğrulama |
| `sklearn.linear_model` | Lojistik Regresyon |
| `sklearn.neighbors` | K-En Yakın Komşu (KNN) |
| `sklearn.tree` | Karar Ağacı |
| `sklearn.metrics` | Karışıklık matrisi, sınıflandırma raporu, ROC eğrisi |

In [ ]:
# Gerekli kütüphaneler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, 
    roc_curve, auc, RocCurveDisplay
)
from sklearn.preprocessing import label_binarize

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("Kütüphaneler başarıyla yüklendi!")

### 2.2 Iris Veri Seti

**Iris (Süsen Çiçeği) veri seti**, makine öğrenmesinin en klasik veri setidir. Ronald Fisher tarafından 1936'da yayınlanmıştır.

- **150 örnek**, 3 tür iris çiçeği (setosa, versicolor, virginica — her birinden 50'şer)
- **4 özellik:** Çanak yaprak uzunluğu/genişliği, taç yaprak uzunluğu/genişliği (cm)

Scikit-learn'de hazır olarak gelir — indirmeye gerek yok.

> Aşağıdaki kodda veri setini yüklüyoruz ve temel bilgilerine bakıyoruz.

In [ ]:
# Iris veri setini yükle
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='tür')

print("Veri seti boyutu:", X.shape)
print("\nÖzellik adları:", list(iris.feature_names))
print("\nSınıf dağılımı:")
for i, name in enumerate(iris.target_names):
    print(f"  {name}: {(y == i).sum()} örnek")

### 2.3 İkili Sınıflandırma İçin Veri Filtreleme

Iris veri setinde 3 sınıf var ama biz **ikili sınıflandırma** öğreneceğiz. Bu nedenle:
- `setosa` sınıfını çıkarıyoruz (çok kolay ayrışır, öğretici değil)
- Sadece **versicolor (0)** ve **virginica (1)** sınıflarını tutuyoruz
- Bu iki sınıf birbirine daha yakın olduğundan, algoritmaların gerçek gücünü test etmiş oluruz

In [ ]:
# İkili sınıflandırma için: sadece versicolor (1) ve virginica (2)
mask = y >= 1
X_binary = X[mask].reset_index(drop=True)
y_binary = (y[mask] - 1).reset_index(drop=True)  # 0: versicolor, 1: virginica

print(f"İkili veri seti boyutu: {X_binary.shape}")
print(f"\nSınıf dağılımı:")
print(f"  0 (versicolor): {(y_binary == 0).sum()}")
print(f"  1 (virginica):  {(y_binary == 1).sum()}")

### 2.4 Eğitim ve Test Setlerine Ayırma

Veriyi **%70 eğitim** ve **%30 test** olarak ikiye bölüyoruz.

- `random_state=42` → Her çalıştırmada aynı bölünmeyi garanti eder (tekrarlanabilirlik)
- `stratify=y_binary` → Her iki sette de sınıf oranlarının aynı kalmasını sağlar

> **Neden bölüyoruz?** Model eğitim verisinde ezberleyebilir (overfitting). Test seti, modelin daha önce hiç görmediği veri üzerindeki gerçek performansını ölçer.

In [ ]:
# Eğitim ve test setlerine ayır
X_train, X_test, y_train, y_test = train_test_split(
    X_binary, y_binary, test_size=0.3, random_state=42, stratify=y_binary
)

print(f"Eğitim seti: {X_train.shape[0]} örnek")
print(f"Test seti:   {X_test.shape[0]} örnek")

## 3. Model Eğitimi

Üç farklı sınıflandırma algoritmasını aynı veri üzerinde eğitip karşılaştıracağız.

### Algoritma Özetleri

| Algoritma | Çalışma Prensibi | Analoji |
|-----------|------------------|--------|
| **Lojistik Regresyon** | Doğrusal bir sınır çizer. Sigmoid fonksiyonu ile 0-1 arası olasılık üretir. | Bir çizgiyle sınıfları ayırmak |
| **KNN (k=5)** | Yeni veriye en yakın 5 komşuya bakar, çoğunluğun sınıfını atar. | "Arkadaşlarına göre seni tanırım" |
| **Karar Ağacı** | If-else kuralları zinciri oluşturur. Her dallanmada en iyi soruyu sorar. | 20 Soruda Bul oyunu |

Her model için 3 adım uyguluyoruz:
1. `.fit(X_train, y_train)` → Eğitim verisinden öğren
2. `.predict(X_test)` → Test verisinde sınıf tahmini yap (0 veya 1)
3. `.predict_proba(X_test)[:, 1]` → Sınıf 1 (virginica) olma olasılığını hesapla

### 3.1 Lojistik Regresyon

İsminde "regresyon" geçse de aslında bir **sınıflandırma** algoritmasıdır. Çıktıyı sigmoid fonksiyonundan geçirerek 0-1 arası bir olasılık üretir. Eşik değeri (genellikle 0.5) üzerindekiler sınıf 1, altındakiler sınıf 0 olarak atanır.

In [ ]:
# Model 1: Lojistik Regresyon
lr_model = LogisticRegression(random_state=42, max_iter=200)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]

print("Lojistik Regresyon — Test Doğruluğu:", f"{lr_model.score(X_test, y_test):.4f}")

### 3.2 K-En Yakın Komşu (KNN)

KNN, **tembel öğrenici (lazy learner)** olarak bilinir — eğitim aşamasında hiçbir şey öğrenmez, tüm veriyi hafızada tutar. Tahmin yaparken yeni veriye en yakın `k` komşuya bakar ve çoğunluk oylamasıyla sınıf atar.

**k değeri seçimi:**
- Çok küçük k (örn. 1): Gürültüye hassas, aşırı öğrenme riski
- Çok büyük k (örn. 50): Karar sınırları çok düz, yetersiz öğrenme riski
- Genellikle tek sayı seçilir (beraberlik olmasın diye): k=3, 5, 7

In [ ]:
# Model 2: K-En Yakın Komşu (KNN)
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
knn_pred = knn_model.predict(X_test)
knn_proba = knn_model.predict_proba(X_test)[:, 1]

print("KNN (k=5) — Test Doğruluğu:", f"{knn_model.score(X_test, y_test):.4f}")

### 3.3 Karar Ağacı (Decision Tree)

Karar ağacı, veriyi **if-else kurallarıyla** bölen sezgisel bir algoritmadır. Her düğümde "Hangi soruyu sorarsam sınıfları en iyi ayırırım?" sorusunu sorar.

- `max_depth=3` → Ağacın maksimum 3 seviye derinliğe ulaşmasına izin veriyoruz (aşırı öğrenmeyi önler)
- Ölçekleme gerektirmez (mesafe hesabı yapmaz)
- En büyük avantajı: İnsan tarafından **yorumlanabilir** olmasıdır

In [ ]:
# Model 3: Karar Ağacı
dt_model = DecisionTreeClassifier(random_state=42, max_depth=3)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
dt_proba = dt_model.predict_proba(X_test)[:, 1]

print("Karar Ağacı — Test Doğruluğu:", f"{dt_model.score(X_test, y_test):.4f}")

## 4. Karışıklık Matrisi ve Sınıflandırma Raporu

### 4.1 Karışıklık Matrisi Nedir?

Modelin tahminlerini 4 kategoride özetler:

|  | Tahmin: Versicolor | Tahmin: Virginica |
|--|--------------------|-----------------|
| **Gerçek: Versicolor** | **TN** (Doğru Negatif) | **FP** (Yanlış Pozitif) |
| **Gerçek: Virginica** | **FN** (Yanlış Negatif) | **TP** (Doğru Pozitif) |

- **Köşegen değerler** (TN, TP): Doğru tahminler — ne kadar büyükse o kadar iyi
- **Köşegen dışı değerler** (FP, FN): Hatalar — ne kadar küçükse o kadar iyi

Aşağıda 3 modelin karışıklık matrislerini yan yana göreceğiz.

In [ ]:
# Her model için karışıklık matrisi — yan yana 3 ısı haritası
models = {
    'Lojistik Regresyon': lr_pred,
    'KNN': knn_pred,
    'Karar Ağacı': dt_pred
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, pred) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Versicolor', 'Virginica'],
                yticklabels=['Versicolor', 'Virginica'])
    ax.set_title(f'{name}\nKarışıklık Matrisi')
    ax.set_xlabel('Tahmin')
    ax.set_ylabel('Gerçek')

plt.tight_layout()
plt.show()

### 4.2 Sınıflandırma Raporu

`classification_report` fonksiyonu her sınıf için **Precision**, **Recall**, **F1-Score** ve **Support** (o sınıftaki gerçek örnek sayısı) değerlerini gösterir.

Rapordaki satırları nasıl okumalı:
- **Precision (Kesinlik):** Bu sınıfı tahmin ettiğimde ne kadar haklıyım?
- **Recall (Duyarlılık):** Bu sınıftan kaç tanesini yakaladım?
- **F1-Score:** Precision ile Recall'un dengesi (ikisi de yüksekse F1 de yüksek)
- **macro avg:** Sınıfların ağırlıksız ortalaması
- **weighted avg:** Sınıfların örnek sayısına göre ağırlıklı ortalaması

In [ ]:
# Sınıflandırma raporları — 3 model karşılaştırma
target_names = ['Versicolor', 'Virginica']

for name, pred in models.items():
    print(f"\n{'='*50}")
    print(f"{name} — Sınıflandırma Raporu")
    print('='*50)
    print(classification_report(y_test, pred, target_names=target_names))

## 5. ROC Eğrisi ve AUC

### ROC Eğrisi Ne Gösterir?

**ROC (Receiver Operating Characteristic)** eğrisi, modelin farklı **eşik değerlerinde** nasıl performans gösterdiğini görselleştirir.

- **X ekseni — Yanlış Pozitif Oranı (FPR):** Versicolor olanların kaçına yanlışlıkla Virginica dedik?
- **Y ekseni — Doğru Pozitif Oranı (TPR):** Virginica olanların kaçını doğru yakaladık?

### AUC (Area Under Curve)

| AUC Değeri | Yorumu |
|-----------|--------|
| **0.90 – 1.00** | Mükemmel |
| **0.80 – 0.90** | Çok iyi |
| **0.70 – 0.80** | İyi |
| **0.60 – 0.70** | Orta |
| **0.50 – 0.60** | Zayıf (rastgeleden biraz iyi) |
| **0.50** | Rastgele tahmin (yazı-tura) |

> **Kural:** Eğri sol üst köşeye ne kadar yakınsa, model o kadar başarılıdır. Kesikli çizgi (AUC=0.5) rastgele tahmini temsil eder.

In [ ]:
# ROC eğrileri — 3 model aynı grafikte
plt.figure(figsize=(10, 7))

probas = {
    'Lojistik Regresyon': lr_proba,
    'KNN': knn_proba,
    'Karar Ağacı': dt_proba
}

colors = ['#2196F3', '#4CAF50', '#FF9800']

for (name, proba), color in zip(probas.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, linewidth=2, 
             label=f'{name} (AUC = {roc_auc:.3f})')

# Rastgele sınıflandırıcı çizgisi
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Rastgele (AUC = 0.500)')

plt.xlabel('Yanlış Pozitif Oranı (FPR)')
plt.ylabel('Doğru Pozitif Oranı (TPR)')
plt.title('ROC Eğrileri — Model Karşılaştırması')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Çapraz Doğrulama ile Model Karşılaştırması

### Neden Çapraz Doğrulama?

Tek bir eğitim/test bölünmesinde model **şanslı** veya **şanssız** olabilir. **Çapraz doğrulama (Cross Validation)** bunu çözer:

1. Veri **k parçaya** bölünür (biz k=5 kullanacağız)
2. Her turda farklı bir parça test seti olarak kullanılır
3. 5 farklı doğruluk skoru elde edilir
4. Ortalaması ve standart sapması raporlanır

```
Tur 1: [TEST] [Eğit] [Eğit] [Eğit] [Eğit]
Tur 2: [Eğit] [TEST] [Eğit] [Eğit] [Eğit]
Tur 3: [Eğit] [Eğit] [TEST] [Eğit] [Eğit]
Tur 4: [Eğit] [Eğit] [Eğit] [TEST] [Eğit]
Tur 5: [Eğit] [Eğit] [Eğit] [Eğit] [TEST]
```

> **Standart sapma** düşükse model tutarlı; yüksekse verideki bölünmeye çok bağımlı demektir.

In [ ]:
# 5 katlı çapraz doğrulama — her model için
cv_models = {
    'Lojistik Regresyon': LogisticRegression(random_state=42, max_iter=200),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Karar Ağacı': DecisionTreeClassifier(random_state=42, max_depth=3)
}

cv_results = {}
print("5 Katlı Çapraz Doğrulama Sonuçları")
print("=" * 55)

for name, model in cv_models.items():
    scores = cross_val_score(model, X_binary, y_binary, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:25s} | Ortalama: {scores.mean():.4f} +/- {scores.std():.4f}")

# Kutu grafiği — her modelin 5 turluk skorlarını gösterir
plt.figure(figsize=(10, 6))
plt.boxplot(cv_results.values(), labels=cv_results.keys(), patch_artist=True,
            boxprops=dict(facecolor='lightblue'))
plt.ylabel('Doğruluk (Accuracy)')
plt.title('Çapraz Doğrulama — Model Karşılaştırması')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Karar Sınırları Görselleştirmesi (2D)

### Karar Sınırı Nedir?

Her sınıflandırma algoritması, özellik uzayında bir **karar sınırı** çizer. Bu sınırın bir tarafı "Sınıf 0", diğer tarafı "Sınıf 1" olarak etiketlenir.

- **Lojistik Regresyon:** Düz bir çizgi çizer (doğrusal sınıflandırıcı)
- **KNN:** Eğri ve düzensiz bir sınır oluşturur (veri noktalarının dağılımına göre)
- **Karar Ağacı:** Eksenlere paralel dikdörtgensel bölgeler oluşturur (her karar bir özelliğe dayanır)

Görselleştirmek için sadece **2 özellik** kullanmamız gerekiyor (ekran 2 boyutlu). En ayırt edici özellikler olan **taç yaprak uzunluğu** ve **taç yaprak genişliğini** seçiyoruz.

> Arka plan rengi modelin o bölge için tahminini, noktalar ise gerçek veri noktalarını gösterir.

In [ ]:
from matplotlib.colors import ListedColormap

# Sadece 2 özellik seç: petal length ve petal width
X_2d = X_binary[['petal length (cm)', 'petal width (cm)']].values
y_2d = y_binary.values

# Modelleri 2D veri üzerinde eğit
models_2d = {
    'Lojistik Regresyon': LogisticRegression(random_state=42, max_iter=200),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Karar Ağacı': DecisionTreeClassifier(random_state=42, max_depth=3)
}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
cmap_bold = ListedColormap(['#FF0000', '#0000FF'])

for ax, (name, model) in zip(axes, models_2d.items()):
    model.fit(X_2d, y_2d)
    
    # Karar sınırı ızgarası oluştur
    h = 0.02
    x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
    y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Karar bölgelerini renklendir
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_light)
    # Gerçek veri noktalarını çiz
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y_2d, cmap=cmap_bold, 
               edgecolors='k', s=50, alpha=0.8)
    ax.set_xlabel('Taçyaprak Uzunluğu (cm)')
    ax.set_ylabel('Taçyaprak Genişliği (cm)')
    ax.set_title(f'{name}\nKarar Sınırları')

plt.tight_layout()
plt.show()

## Özet ve Önemli Çıkarımlar

Bu defterde sınıflandırmanın temellerini uygulamalı olarak öğrendik:

### Adımlar
1. **Iris veri seti** yükleme ve ikili sınıflandırma için filtreleme
2. **3 farklı algoritma** ile model eğitimi (Lojistik Regresyon, KNN, Karar Ağacı)
3. **Karışıklık matrisi** ile hataları görselleştirme
4. **Sınıflandırma raporu** ile detaylı metrikler (Precision, Recall, F1)
5. **ROC eğrisi ve AUC** ile eşikten bağımsız karşılaştırma
6. **Çapraz doğrulama** ile güvenilir performans ölçümü
7. **Karar sınırları** ile algoritmaların farklılıklarını görme

### Model Karşılaştırma Özeti

| Algoritma | Avantaj | Dezavantaj | Ne Zaman Kullan? |
|-----------|---------|-----------|------------------|
| **Lojistik Regresyon** | Hızlı, yorumlanabilir, olasılık verir | Doğrusal olmayan ilişkilerde zayıf | Başlangıç modeli (baseline) olarak her zaman |
| **KNN** | Basit, eğitim gerektirmez | Yavaş (büyük veride), ölçekleme şart | Küçük veri setlerinde, hızlı prototip |
| **Karar Ağacı** | Görsel, yorumlanabilir | Overfitting riski yüksek | İş kuralları çıkarmak gerektiğinde |

### Alıştırmalar
1. KNN'de `n_neighbors` değerini 1, 3, 7, 11, 15 yaparak doğruluğu karşılaştırın
2. Karar Ağacı'nda `max_depth` parametresini kaldırarak overfitting gözlemleyin
3. Iris'in 3 sınıfını birden kullanarak **çoklu sınıflandırma** deneyin
4. Karar sınırlarını diğer 2 özellik çiftiyle (sepal length/width) tekrar çizdirin

### Sonraki Adım
Bir sonraki defterde bu bilgileri **Kaggle Titanic** yarışmasında uygulayacağız!

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

© 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>